# NER Pipeline — Cross-Edition Entity Index

Gazetteer-based Named Entity Recognition across all eight Popol Wuj editions.

**Strategy:** canonical entity list from `temas/topics.xml` + `textos/colop-glossary.txt`
+ Ximénez annotation surface-form variants. Greedy longest-match n-gram lookup (n = 1–4)
with exact-then-fuzzy (apostrophe+diacritic-stripped) fallback.

**Location granularity by edition:**

| Language | Editions | Unit |
|----------|----------|------|
| K'iche' | ajtzibab, christenson, colop, christenson_ximenez, ximenez | paragraph (doc_id) |
| Spanish | recinos | sentence (parte.capit.sent) |
| English | tedlock | sentence (part.sent) |
| English | christenson_english_prose | chapter (chap_num) |

**Outputs written to this directory:**
- `GAZETTEER.csv` — entity × language × surface-form
- `{edition}-NER.csv` × 8 — one row per mention
- `ENTITY_CROSSMAP.csv` — aggregated mention counts (loaded by the Streamlit page)


In [ ]:
import sys
from pathlib import Path

_here = Path().resolve()
sys.path.insert(0, str(_here))

import ner_pipeline as ner
import pandas as pd


## Step 1 — Build gazetteer and run all editions

In [ ]:
# Run the full pipeline — re-run any time source data changes.
ALL = ner.run(out_dir=_here, verbose=True)


## Step 2 — Inspect outputs

In [ ]:
gaz  = pd.read_csv(_here / 'GAZETTEER.csv')
xmap = pd.read_csv(_here / 'ENTITY_CROSSMAP.csv')

print(f"Gazetteer: {len(gaz)} form entries, {gaz['entity_id'].nunique()} entities")
print()
print("Forms per language:")
print(gaz.groupby('lang')['form'].count().to_string())


In [ ]:
# Top 30 entities by total mentions across all editions
top = (xmap.groupby(['entity_id', 'label', 'type'])['mention_count']
       .sum()
       .sort_values(ascending=False)
       .head(30)
       .reset_index())
top.style.background_gradient(cmap='YlOrRd', subset=['mention_count'])


In [ ]:
# Cross-edition heatmap (pivot: entity label × edition)
pivot = xmap.pivot_table(
    index='label', columns='edition', values='mention_count', fill_value=0
)
pivot = pivot.loc[pivot.sum(axis=1).sort_values(ascending=False).head(40).index]
pivot


In [ ]:
# Entities present in all 8 editions
all_eds = set(xmap['edition'].unique())
ubiquitous = (xmap.groupby('entity_id')['edition']
              .apply(set)
              .pipe(lambda s: s[s.apply(lambda x: x == all_eds)].index))
labels = (xmap[xmap['entity_id'].isin(ubiquitous)]
          .drop_duplicates('entity_id')['label'].tolist())
print(f"Entities in all 8 editions ({len(ubiquitous)}): {labels}")


## Per-edition inspection

In [ ]:
edition = 'colop'  # change to explore other editions

ner_df = pd.read_csv(_here / f'{edition}-NER.csv')
print(f"{edition}: {len(ner_df)} mentions, {ner_df['entity_id'].nunique()} unique entities")
print()
print("Top 20 entities by mention count:")
print(
    ner_df.groupby(['entity_id', 'label', 'type'])['ngram']
    .count()
    .sort_values(ascending=False)
    .head(20)
    .to_string()
)


In [ ]:
entity = 'JUNAJPU'
sample = ner_df[ner_df['entity_id'] == entity].head(20)
print(f"First 20 mentions of {entity} in {edition}:")
print(sample[['ohco_path', 'ngram', 'match_type']].to_string())
